# Chapter 13 -- Transparency Through Decomposition

Companion notebook for `affinity/book/chapters/CH13_transparency.tex`.

Demonstrates the four-equation diagnostic of §13.1 on three contrast cases:

1. Synthetic symmetric data + full-bilinear predictor (spurious asymmetric energy).
2. Synthetic asymmetric pairwise data + full predictor (faithful asymmetric energy).
3. A trained `StdAttnDecomposableTransformer` from `tabkernels.audits._models` (real architecture).
4. Practical workflow: end-to-end on Breast Cancer Wisconsin (sklearn).

Runtime budget: < 10 minutes via papermill.

In [ ]:
# Cell 1: setup
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from tabkernels.audits._helpers import train_pairwise
from tabkernels.audits._models import (
    KernelAttention,
    PairwisePredictor,
    StdAttnDecomposableTransformer,
)
from tabkernels.audits.pairwise import make_pairwise_data
from tabkernels.core.decomposition import energy_split
from tabkernels.transparency import (
    decompose_attention,
    inspect_attention,
    kernel_energy_split,
    recovery_cosine,
)

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

def _find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.basename(p) == 'similarity-hierarchy-research':
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join('..', '..'))

REPO_ROOT = _find_repo_root()
FIGURES_DIR = os.path.join(REPO_ROOT, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print('FIGURES_DIR:', FIGURES_DIR)
T0 = time.time()

## Case 1 -- Synthetic symmetric data: spurious asymmetric energy

Train a full-bilinear `PairwisePredictor(variant='full')` on a synthetic
regression task with purely symmetric ground-truth pairwise structure (no
skew kernel in the data-generating process).  The diagnostic should report
$\\alpha_A$ small but non-zero with $\\cos_A$ small as well -- the model
spends asymmetric capacity on noise.

In [ ]:
# Cell 2: spurious case -- symmetric ground truth, Std-style kernel.
# A KernelAttention(variant='std') has independent W_Q, W_K, so its
# B = W_Q^T W_K is generically asymmetric even when fit to symmetric data.
torch.manual_seed(2)
data_sym = make_pairwise_data(N=80, D=6, regime='symmetric', noise=0.1, seed=0)
Z = torch.tensor(data_sym['Z'], dtype=torch.float32)
y_sym = Z @ torch.tensor(data_sym['A_true'], dtype=torch.float32) @ Z.T
y_sym = y_sym.diag()  # use the diagonal as a regression target
A_true_sym = torch.tensor(data_sym['A_true'], dtype=torch.float32)

model_sym = KernelAttention(d_in=Z.shape[1], d_emb=8, variant='std')
y_sym_target = y_sym.unsqueeze(-1)
opt = torch.optim.Adam(model_sym.parameters(), lr=5e-3)
for step in range(300):
    opt.zero_grad()
    pred = model_sym.predict(Z, Z, y_sym_target, mask_diag=True).squeeze(-1)
    loss = ((pred - y_sym) ** 2).mean()
    loss.backward()
    opt.step()

# Apply the diagnostic.
parts = decompose_attention(model_sym)
alpha_S, alpha_A = kernel_energy_split(parts['B'])
B_learned_sym = parts['B'][0]
cos_S, cos_A = recovery_cosine(B_learned_sym, A_true_sym)
print('Case 1 -- synthetic symmetric ground truth, Std-style attention')
print(f'  alpha_S = {alpha_S:.3f}   alpha_A = {alpha_A:.3f}')
print(f'  cos_S   = {cos_S:.3f}   cos_A   = {cos_A:.3f}')
print('  reading: alpha_A non-trivial but cos_A small -> spurious asymmetric energy.')

SPURIOUS_ALPHA_A = float(alpha_A)
SPURIOUS_COS_A = float(cos_A)
assert SPURIOUS_ALPHA_A > 0.05, f'sym-data alpha_A too small to demonstrate: {SPURIOUS_ALPHA_A}'
assert abs(SPURIOUS_COS_A) < 0.95, f'sym-data cos_A unexpectedly close to 1: {SPURIOUS_COS_A}'

## Case 2 -- Synthetic asymmetric pairwise: faithful asymmetric energy

Same data-generating recipe but with $A$ unconstrained, so the truth has
non-zero skew energy.  The full predictor should reach $\\alpha_A$
matching the truth, with $\\cos_A \\approx 1$.

In [ ]:
# Cell 3: faithful case -- asymmetric pairwise data, full predictor.
torch.manual_seed(0)
data_asym = make_pairwise_data(N=120, D=6, regime='asymmetric', noise=0.05, seed=0)
Z2 = data_asym['Z']
R2 = data_asym['R']
A_true_asym = torch.tensor(data_asym['A_true'], dtype=torch.float32)
train_mask2 = np.ones((Z2.shape[0], Z2.shape[0]), dtype=np.float32)

model_asym = PairwisePredictor(d_in=Z2.shape[1], variant='full')
train_pairwise(model_asym, Z2, R2, train_mask2, epochs=600, lr=2e-2)

B_learned_asym = model_asym.kernel_matrix().detach().cpu()
alpha_S2, alpha_A2 = energy_split(B_learned_asym)
cos_S2, cos_A2 = recovery_cosine(B_learned_asym, A_true_asym)
true_alpha_S, true_alpha_A = energy_split(A_true_asym)

print('Case 2 -- synthetic asymmetric pairwise ground truth, full predictor')
print(f'  alpha_S = {alpha_S2:.3f}   alpha_A = {alpha_A2:.3f}')
print(f'  cos_S   = {cos_S2:.3f}   cos_A   = {cos_A2:.3f}')
print(f'  truth   alpha_S = {true_alpha_S:.3f}   alpha_A = {true_alpha_A:.3f}')
print('  reading: alpha_A close to truth and cos_A close to 1 -> faithful.')

FAITHFUL_ALPHA_A = float(alpha_A2)
FAITHFUL_COS_A = float(cos_A2)
assert abs(FAITHFUL_ALPHA_A - true_alpha_A) < 0.10
assert FAITHFUL_COS_A > 0.95

## Case 3 -- Inspecting a real `StdAttnDecomposableTransformer`

After a brief training pass on a small synthetic ICL task, the diagnostic
reads off the asymmetric energy of the in-context attention head.  No
oracle target is available, so we report only $(\\alpha_S, \\alpha_A)$
and the post-hoc projection metrics; the full-vs-sym gap is the third
reading from the decision rule.  This case exercises
`inspect_attention` end-to-end.

In [ ]:
# Cell 4: real architecture -- StdAttnDecomposableTransformer.
torch.manual_seed(0)
d_x, d_model, n_heads, n_layers, n_classes = 6, 16, 4, 1, 2
n_ctx, n_q = 16, 4

icl_model = StdAttnDecomposableTransformer(
    d_x=d_x, d_model=d_model, n_heads=n_heads, n_layers=n_layers,
    n_classes=n_classes,
).to(DEVICE)

B_episodes = 32
X_ctx = torch.randn(B_episodes, n_ctx, d_x, device=DEVICE)
y_ctx = (torch.randn(B_episodes, n_ctx, device=DEVICE) > 0).long()
X_q   = torch.randn(B_episodes, n_q, d_x, device=DEVICE)
y_q   = (torch.randn(B_episodes, n_q, device=DEVICE) > 0).long()

opt = torch.optim.Adam(icl_model.parameters(), lr=1e-3)
icl_model.train()
for step in range(40):
    opt.zero_grad()
    logits = icl_model(X_ctx, y_ctx, X_q)
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, n_classes), y_q.reshape(-1))
    loss.backward()
    opt.step()
icl_model.eval()

class _Wrap(nn.Module):
    def __init__(self, base, X_ctx, y_ctx):
        super().__init__()
        self.base = base
        self.register_buffer('X_ctx', X_ctx)
        self.register_buffer('y_ctx', y_ctx)
    def forward(self, X_q):
        return self.base(self.X_ctx, self.y_ctx, X_q)

wrapped = _Wrap(icl_model, X_ctx, y_ctx).to(DEVICE)
report = inspect_attention(wrapped, X_q, target=None, y=y_q)
print('Case 3 -- StdAttnDecomposableTransformer after 40 steps')
print(f'  per-head count    : {len(report["per_head"])}')
print(f'  alpha_S (mean)    : {report["alpha_S"]:.3f}')
print(f'  alpha_A (mean)    : {report["alpha_A"]:.3f}')
print(f'  full_acc          : {report["full_acc"]:.3f}')
print(f'  sym-only_acc      : {report["sym_only_acc"]:.3f}')
print(f'  asym-only_acc     : {report["asym_only_acc"]:.3f}')
print(f'  full-vs-sym gap   : {report["full_acc"] - report["sym_only_acc"]:+.3f}')
REAL_ALPHA_A = float(report['alpha_A'])
assert 0.0 < REAL_ALPHA_A < 1.0

## Case 4 -- Practical workflow on Breast Cancer

End-to-end demonstration on a real (small) tabular classification dataset:
Breast Cancer Wisconsin (`sklearn.datasets.load_breast_cancer`, 569 rows,
30 features, binary).  We train a `KernelAttention(variant='std')` on a
subset, then run the four-equation diagnostic and post-hoc projections.

In [ ]:
# Cell 5: practical demo -- Breast Cancer Wisconsin.
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

torch.manual_seed(0)
X_np, y_np = load_breast_cancer(return_X_y=True)
X_np = StandardScaler().fit_transform(X_np)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

perm = torch.randperm(X.shape[0])
n_train = 200
n_test = 150
X_tr, X_te = X[perm[:n_train]], X[perm[n_train:n_train+n_test]]
y_tr, y_te = y[perm[:n_train]], y[perm[n_train:n_train+n_test]]

real_model = KernelAttention(d_in=X.shape[1], d_emb=8, variant='std')
y_tr_f = y_tr.float()
opt = torch.optim.Adam(real_model.parameters(), lr=5e-3)
for step in range(120):
    opt.zero_grad()
    pred = real_model.predict(X_tr, X_tr, y_tr_f.unsqueeze(-1), mask_diag=True).squeeze(-1)
    loss = ((pred - y_tr_f) ** 2).mean()
    loss.backward()
    opt.step()

parts = decompose_attention(real_model)
alpha_S_bc, alpha_A_bc = kernel_energy_split(parts['B'])
print('Case 4 -- KernelAttention(std) on Breast Cancer Wisconsin')
print(f'  d_x = {X.shape[1]}, n_train = {n_train}')
print(f'  per-head bilinear forms : {parts["B"].shape}')
print(f'  alpha_S = {alpha_S_bc:.3f}   alpha_A = {alpha_A_bc:.3f}')

# Post-hoc projection on the rectangular query/test gram.  We project the
# *bilinear form* B (square) rather than the rectangular score matrix S;
# this is the rectangular-query analogue of Eq. 13.2 and equivalent on
# square inputs (cf. ch:13:psd-vs-general).
def _kpredict_with_B(B, X_q, X_t, y_t, scale):
    S = (X_q @ B @ X_t.T) / scale
    W = torch.softmax(S, dim=-1)
    return W @ y_t

real_model.eval()
B_full = (real_model.W_Q.weight.T @ real_model.W_K.weight).detach()
B_S = (B_full + B_full.T) / 2
B_A = (B_full - B_full.T) / 2
import math
scale = math.sqrt(real_model.d_emb)
accs = {}
with torch.no_grad():
    for mode, B_use in (('full', B_full), ('sym', B_S), ('asym', B_A)):
        pred = _kpredict_with_B(B_use, X_te, X_tr,
                                 y_tr.float().unsqueeze(-1), scale).squeeze(-1)
        accs[mode] = float(((pred > 0.5) == y_te).float().mean())
print(f'  full accuracy       : {accs["full"]:.3f}')
print(f'  sym-only accuracy   : {accs["sym"]:.3f}')
print(f'  asym-only accuracy  : {accs["asym"]:.3f}')
print(f'  full-vs-sym gap     : {accs["full"] - accs["sym"]:+.3f}')
print('  reading: small full-vs-sym gap -> asymmetric capacity is not')
print('           doing predictive work; sym-only kernel suffices.')

## Figure 13.1 -- Three contrast cases

In [ ]:
# Cell 6: Figure 13.1 -- bar chart contrasting alpha_A and cos_A.
labels = ['symmetric synth\n(spurious)',
          'asymmetric synth\n(faithful)',
          'real arch\n(StdAttnDec)',
          'breast cancer\n(KernelAttn-std)']
alpha_As = [SPURIOUS_ALPHA_A, FAITHFUL_ALPHA_A, REAL_ALPHA_A, float(alpha_A_bc)]
cos_As   = [SPURIOUS_COS_A,   FAITHFUL_COS_A,   float('nan'),  float('nan')]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
ax = axes[0]
ax.bar(range(4), alpha_As,
       color=['#4477AA', '#EE6677', '#228833', '#CCBB44'])
ax.set_xticks(range(4))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel(r'$\alpha_A$ (asymmetric energy fraction)')
ax.set_title('Asymmetric energy (Eq. 13.3)')
ax.set_ylim(0, 1.0)
for i, v in enumerate(alpha_As):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
ax = axes[1]
x = np.arange(4)
vals_for_plot = [v if not np.isnan(v) else 0.0 for v in cos_As]
ax.bar(x, vals_for_plot, color=['#4477AA', '#EE6677', '#228833', '#CCBB44'])
ax.axhline(0, color='gray', linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel(r'$\cos_A$ (cosine to truth)')
ax.set_title('Faithfulness (Eq. 13.4)')
ax.set_ylim(-0.2, 1.2)
for i, v in enumerate(cos_As):
    if np.isnan(v):
        ax.text(i, 0.05, 'no oracle', ha='center', fontsize=8, style='italic')
    else:
        ax.text(i, v + 0.04, f'{v:.2f}', ha='center', fontsize=9)
fig.suptitle('Figure 13.1 -- Decomposition diagnostic across four cases')
fig.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_13_01_decomposition_diagnostic.pdf')
fig.savefig(out_path)
plt.show()
print('saved:', out_path)

## Summary and runtime budget

In [ ]:
# Cell 7: end-of-notebook summary.
elapsed = time.time() - T0
print('=' * 60)
print(f'Chapter 13 notebook -- all checks passed (runtime {elapsed:.1f} s)')
print('=' * 60)
print(f'Spurious case  : alpha_A = {SPURIOUS_ALPHA_A:.3f}, cos_A = {SPURIOUS_COS_A:.3f}')
print(f'Faithful case  : alpha_A = {FAITHFUL_ALPHA_A:.3f}, cos_A = {FAITHFUL_COS_A:.3f}')
print(f'Real-arch case : alpha_A = {REAL_ALPHA_A:.3f}  (no oracle target)')
print(f'Breast Cancer  : alpha_A = {alpha_A_bc:.3f}, full-vs-sym gap = '
      f'{accs["full"] - accs["sym"]:+.3f}')
print()
print('Diagnostic decision rule (Table 13.3):')
print('  alpha_A approx 0       -> symmetric kernel suffices')
print('  alpha_A >> 0, cos_A=1  -> faithful asymmetric')
print('  alpha_A >> 0, cos_A=0  -> spurious asymmetric energy')
assert elapsed < 600, f'notebook over runtime budget: {elapsed:.1f}s'